# Length weighting and entropy need context

Day 24 · Chapter 14 · CPU worked notebook

Read Chapter14 sections14.3–14.5. Predict: do length2 and length8 responses get equal total weight under a global token mean? Does low entropy by itself mean learning failed?

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: length entropy diagnostics](../figures/chapter-14/day-24-02_length_entropy_diagnostics-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.optimization_diagnostics_lab import length_weights
lengths = [2, 8]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for reduction, offset in (("response", -.15), ("token", .15)):
    weights = length_weights(lengths, reduction)
    axes[0].bar(torch.arange(2).numpy()+offset, weights, width=.3, label=reduction)
axes[0].set(xticks=[0,1], xticklabels=["2-token response","8-token response"],
            ylabel="Total nominal response weight")
axes[0].legend()
strength = torch.linspace(0, 8, 101, dtype=torch.float64)
p = torch.stack((strength, torch.zeros_like(strength), torch.zeros_like(strength)), -1).softmax(-1)
entropy = -(p*p.log()).sum(-1)
axes[1].plot(strength, entropy)
axes[1].set(xlabel="Logit of one favored outcome", ylabel="Conditional entropy (nats)")
fig.tight_layout()
plt.show()

Reference explanation: response means give each response0.5; token means give0.2 and0.8. Entropy declines whether the favored action is correct or incorrect.

Exercise: construct two policies with identical entropy but different strict accuracy.

In [ ]:
# Reference solution: permuting outcome labels preserves entropy.
correct_favored = torch.tensor([.9, .05, .05], dtype=torch.float64)
wrong_favored = torch.tensor([.05, .9, .05], dtype=torch.float64)
H = lambda p: -(p*p.log()).sum()
torch.testing.assert_close(H(correct_favored), H(wrong_favored))
print("Same entropy:", float(H(correct_favored)),
      "different accuracy:", float(correct_favored[0]), float(wrong_favored[0]))
print("Per-token coefficients:", (length_weights(lengths)/torch.tensor(lengths)).tolist())

Controlled change: add a length20 response and compare both reductions. Evidence boundary: coefficient geometry does not prove the trained policy will get longer. Actual gradients, EOS, reward and advantage signs matter. Entropy alone cannot distinguish confident success from collapse.